In [ ]:
# Compare modelled Temperature (temp) with observed temperature (2015)

from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cmocean
from scipy.spatial import cKDTree

# ── Load model output ─────────────────────────────────────────────────────
MODEL_FILE  = Path('/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_01/'
                   'yearly_dws_500m.3d.2015_effective_fetch_Q.nc')
ds = xr.open_dataset(MODEL_FILE)

Marsdiep_ts = 'Jetty_ts.csv'
CHLA_VAR = 'Chla'
ELEV_VAR = 'elev'
Bathymetry_VAR = 'bathymetry'  # Preferred name for bathymetry variable; will try alternatives if not found.

Benthic_POC_ts = '20100215_PAM_overview_1974_2009i.xlsx'

# Chla layer index to inspect and neighboring layers.
CHLA_LAYER_INDEX = 5  # top=11, bottom=1 in your convention

ROLLING_WINDOW = None  # e.g., 3 for smoothing, or None

#PP_csv_path = Validation_DATA_DIR / Marsdiep_PP_ts

# Parameters for model-observation comparison
MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention

USE_DAILY_MEAN = True  # If True, use daily mean of model output; if False, use instantaneous values.
# The time series data for Marsdiep is available from the NIOZ Dataverse at http://doi.org/10.25850/nioz/7b.b.5j

In [ ]:
# List of variables for analysis and visualization
vars_list = [
    'elev',
    #'temp',
    #'salt',
    #'O2o', 
    'netPPm2',
    'N1p',
    'N3n',
    #'N4n',
    #'N5s',
    #'N6r',
#         'B1c',
#         'Bac',
          'P1c',
#     'P2c',
#     'P3c',
#         'P4c',
#         'P5c',
#         'P6c',
#	      'P1l',
#         'P2l',
#         'P3l',
#         'P4l',
#         'P5l',
#         'P6l',
#         'Z2c',
#         'Z3c',
#         'Z4c',
#         'Z5c',
#         'Z6c',
#         'R1c',
#         'R2c',
#         'R3c',
#          'R6c',
#         'RZc',
#         'Q1c',
#         'Q11c',
#          'Q6c',
          'Chla',
#          'H1c',
#          'H2c',
#          'HNc',
#          'Hac',
          'Y1c',
          'Y2c',
          'Y3c',
#          'Y4c',
          'Y5c',
#          'Yy3c',
#          'K6r',
#          'K16r',
#          'K26r',
#          'K5s',
#          'K15s',
#          'K3n',
#          'K4n',
#         'K13n',
#          'K14n',
#          'K24n',
#          'K1p',
#          'K11p',
#         'K21p',
#          'D1m',
#          'D2m',
#          'O3c',
#          'pCO2',
#          'CO2',
#          'HCO3',
#          'CO3',
#          'pH',
#          'Ac'
#          'G3h'
#          'G13h'
#          'G23h'
#          'G3c'
#          'G13c'
#          'G23c'
#          'G14n'
#          'Acae'
#          'Acan'
#          'DICae'
#          'DICan'
#          'pHae'
#          'pHan'
#          'pCO2ae'
#          'pCO2an'
#          'G3h'
#          'G13h'
          'BP1c', # Benthic diatom
#          'ETW',
          'ESS',
#          'irrenh',
#          'turenh',
'xEPS'      
]

In [ ]:
# Helpers
# Reuse opened dataset if available; otherwise open yearly files.

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None
   
def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    # Keep first occurrence when duplicate time stamps are present.
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da
    
def _find_bathy_name(ds: xr.Dataset, preferred: str) -> str:
    if preferred in ds.variables:
        return preferred
    candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo', 'd', 'water_depth')
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(
        f"Bathymetry variable not found. Tried '{preferred}' and {candidates}. "
        f"Available vars include: {list(ds.variables)[:30]}"
    )

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

In [ ]:
# Goal: show the selected subdomain on a bathymetry map.

# Subdomain index range (Python slice: start inclusive, stop exclusive).
# Salt marsh zone nearby Miedema (unusual discontinuity in derived total Chla)
#X_SLICE = (215, 225)
#Y_SLICE = (133, 136)

# Marsdiep zone
X_SLICE = (75, 78)
Y_SLICE = (95, 98)

# Lauwesoog zone
#X_SLICE = (245, 280)
#Y_SLICE = (135, 150)

# Whole subdomain
#X_SLICE = (1, 320)
#Y_SLICE = (1, 190)


bathy_name = _find_bathy_name(ds, Bathymetry_VAR)
bathy = ds[bathy_name].squeeze(drop=True)

# Find horizontal dims from bathymetry; if a time dim exists, use first timestep.
bathy_dims = list(bathy.dims)
time_like = [d for d in bathy_dims if 'time' in d.lower()]
if time_like:
    bathy = bathy.isel({time_like[0]: 0})
    bathy_dims = [d for d in bathy.dims if d != time_like[0]]
    
if len(bathy_dims) != 2:
    raise ValueError(f'Expected 2D bathymetry after squeezing, got dims {bathy.dims}')
y_dim, x_dim = bathy_dims

ny = bathy.sizes[y_dim]
nx = bathy.sizes[x_dim]
y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= ny):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {ny}')
if not (0 <= x0 < x1 <= nx):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {nx}')

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_plot = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_plot = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_plot.shape == (ny, nx) and y_plot.shape == (ny, nx):
            if np.isfinite(x_plot).all() and np.isfinite(y_plot).all():
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

bathy2d = _to_float(bathy.transpose(y_dim, x_dim).values)

fig, ax = plt.subplots(figsize=(8.6, 6.8))
mesh = ax.pcolormesh(
    x_plot,
    y_plot,
    np.ma.masked_invalid(bathy2d),
    shading='auto',
    cmap='cividis',
)
cbar = fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.03)
units = bathy.attrs.get('units', '')
cbar.set_label(f'{bathy_name} [{units}]' if units else bathy_name)

# Draw the selected subdomain as a closed polygon on the map.
if use_geo:
    x_poly = [x_plot[y0, x0], x_plot[y0, x1 - 1], x_plot[y1 - 1, x1 - 1], x_plot[y1 - 1, x0], x_plot[y0, x0]]
    y_poly = [y_plot[y0, x0], y_plot[y0, x1 - 1], y_plot[y1 - 1, x1 - 1], y_plot[y1 - 1, x0], y_plot[y0, x0]]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')
else:
    x_poly = [x0, x1, x1, x0, x0]
    y_poly = [y0, y0, y1, y1, y0]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')

ax.set_title('Subdomain location on bathymetry map')
ax.set_xlabel('Longitude' if use_geo else x_dim)
ax.set_ylabel('Latitude' if use_geo else y_dim)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Bathymetry variable used: {bathy_name}')
print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')

In [ ]:
# Time series of the surface-layer, domain-mean value of each variable in vars_list.
# Uses the helpers (_find_time_dim, _find_vertical_dim, _drop_duplicate_time,
# _maybe_smooth) and MODEL_SURFACE_LAYER_INDEX from the model-vs-observation cell.

# Values below these limits are masked as invalid (keys are lower-case variable names).
VALID_MIN = {"chla": 0, "netppm2": 0, "xeps": 0, "etw": -10}

fig, axes = plt.subplots(
    len(vars_list),
    1,
    figsize=(14, 3 * len(vars_list)),
    sharex=True,
    constrained_layout=True,
    squeeze=False,
)
axes = axes[:, 0]

skipped = []

for ax, vname in zip(axes, vars_list):
    ax.set_title(vname)
    ax.grid(alpha=0.3)

    if vname not in ds.variables:
        skipped.append((vname, "not in dataset"))
        ax.text(0.5, 0.5, f"'{vname}' not in dataset",
                ha="center", va="center", transform=ax.transAxes)
        continue

    try:
        da = ds[vname].squeeze(drop=True)
        time_dim = _find_time_dim(da)
        z_dim = _find_vertical_dim(da, time_dim)

        # Surface layer for 3D variables
        if z_dim is not None:
            da = da.isel({z_dim: MODEL_SURFACE_LAYER_INDEX})

        da = _drop_duplicate_time(da, time_dim)

        vmin = VALID_MIN.get(vname.lower())
        if vmin is not None:
            da = da.where(da >= vmin)

        spatial_dims = [d for d in da.dims if d != time_dim]
        if not spatial_dims:
            raise ValueError(f"no spatial dimensions in {da.dims}")

        series = da.mean(dim=spatial_dims, skipna=True)
        series = _maybe_smooth(series, time_dim)

    except Exception as e:
        skipped.append((vname, str(e)))
        ax.text(0.5, 0.5, f"Could not plot '{vname}':\n{e}",
                ha="center", va="center", transform=ax.transAxes)
        continue

    ax.plot(series[time_dim].values, series.values, lw=1.4, color="tab:blue")

    units = ds[vname].attrs.get("units", "")
    ax.set_ylabel(f"{vname} [{units}]" if units else vname)

axes[-1].set_xlabel("Time")
fig.suptitle(f"Surface-layer domain mean | {MODEL_FILE.name}")
plt.show()

if skipped:
    print("Skipped variables:")
    for vname, reason in skipped:
        print(f"  {vname}: {reason}")


In [ ]:
import pandas as pd

Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
csv_path = Validation_DATA_DIR / Marsdiep_ts

if not csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {csv_path}')

measurement_df = pd.read_csv(
    csv_path,
    na_values=['NA', ''],
    parse_dates=['timestamp'],
)

print(f'Loaded CSV: {csv_path}')
print(f'Shape: {measurement_df.shape[0]} rows x {measurement_df.shape[1]} columns')
print('Columns:')
print(list(measurement_df.columns))
print('')
print('First 5 rows:')
display(measurement_df.head())



In [ ]:
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.


#Obs_VAR_NAME = 'TSM'
#Model_VAR_NAME = 'ESS'

#Obs_VAR_NAME = 'Daily_PP'
#Model_VAR_NAME = 'netPPm2'

#Obs_VAR_NAME = 'POC'
#Model_VAR_NAME = 'R6c'

#Obs_VAR_NAME = 'NH4'
#Model_VAR_NAME = 'N4n'

Obs_VAR_NAME = 'Chl'
Model_VAR_NAME = 'Chla'

if Model_VAR_NAME not in ds.variables:
    raise KeyError(f"Variable '{Model_VAR_NAME}' not found. Available: {sorted(ds.data_vars)}")

Model_ds = ds[Model_VAR_NAME].squeeze(drop=True)
time_dim = _find_time_dim(Model_ds)

z_dim = _find_vertical_dim(Model_ds, time_dim)   # <-- may be None for 2D variables

# ---------------------------------------------------------
# 1. Handle 2D vs 3D variables
# ---------------------------------------------------------
if z_dim is None:
    # 2D variable: dims = (time, y, x)
    spatial_dims = [d for d in Model_ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected 2D variable with dims (time,y,x), got {Model_ds.dims}")
    y_dim, x_dim = spatial_dims
    Model_sub = Model_ds.isel({y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
                               x_dim: slice(X_SLICE[0], X_SLICE[1])})
else:
    # 3D variable: dims = (time, z, y, x)
    xy_dims = [d for d in Model_ds.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims for {Model_VAR_NAME}, got {xy_dims}')
    y_dim, x_dim = xy_dims

    Model_sub = Model_ds.isel({
        z_dim: MODEL_SURFACE_LAYER_INDEX,
        y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
        x_dim: slice(X_SLICE[0], X_SLICE[1])
    })

# ---------------------------------------------------------
# 2. Compute spatial mean
# ---------------------------------------------------------
model_series = Model_sub.mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Day of year
model_series['doy'] = model_series[time_dim].dt.dayofyear

# ---------------------------------------------------------
# 3. Observations
# ---------------------------------------------------------
if Obs_VAR_NAME not in measurement_df.columns:
    raise KeyError(f"Column {Obs_VAR_NAME} not found in the file.")

obs_df = measurement_df[['timestamp', Obs_VAR_NAME]].dropna(subset=['timestamp', Obs_VAR_NAME]).copy()
obs_df['timestamp'] = pd.to_datetime(obs_df['timestamp'], dayfirst=True, errors='coerce')
obs_df = obs_df.sort_values('timestamp')

if obs_df.empty:
    raise ValueError('No measurements found.')

obs_df['doy'] = obs_df['timestamp'].dt.dayofyear

# ---------------------------------------------------------
# 4. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.scatter(
    obs_df['doy'],
    #obs_df['POC'] * 1000, # for POC to be mg/m3
    obs_df[Obs_VAR_NAME],
    s=22,
    color='tab:orange',
    alpha=0.85,
    label=f'Observed {Obs_VAR_NAME}',
    zorder=3,
)

ax.plot(
    model_series['doy'],
    model_series.values,
    lw=2.0,
    color='tab:blue',
    label=f'Model {Model_VAR_NAME}',
    zorder=4,  # above the scatter (zorder=3)
)


Var_units = Model_ds.attrs.get('units', '')
ax.set_ylabel(f'{Obs_VAR_NAME} [{Var_units}]')
ax.set_xlabel('Day of Year')
ax.set_title(
    f'Model vs observed {Obs_VAR_NAME} | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Number of observation points: {len(obs_df)}')


In [ ]:
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.


#Obs_VAR_NAME = 'TSM'
#Model_VAR_NAME = 'ESS'

Obs_VAR_NAME = 'Daily_PP'
Model_VAR_NAME = 'netPPm2'

#Obs_VAR_NAME = 'POC'
#Model_VAR_NAME = 'R6c'

#Obs_VAR_NAME = 'NH4'
#Model_VAR_NAME = 'N4n'

#Obs_VAR_NAME = 'Chl'
#Model_VAR_NAME = 'Chla'

if Model_VAR_NAME not in ds.variables:
    raise KeyError(f"Variable '{Model_VAR_NAME}' not found. Available: {sorted(ds.data_vars)}")

Model_ds = ds[Model_VAR_NAME].squeeze(drop=True)
time_dim = _find_time_dim(Model_ds)

z_dim = _find_vertical_dim(Model_ds, time_dim)   # <-- may be None for 2D variables

# ---------------------------------------------------------
# 1. Handle 2D vs 3D variables
# ---------------------------------------------------------
if z_dim is None:
    # 2D variable: dims = (time, y, x)
    spatial_dims = [d for d in Model_ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected 2D variable with dims (time,y,x), got {Model_ds.dims}")
    y_dim, x_dim = spatial_dims
    Model_sub = Model_ds.isel({y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
                               x_dim: slice(X_SLICE[0], X_SLICE[1])})
else:
    # 3D variable: dims = (time, z, y, x)
    xy_dims = [d for d in Model_ds.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims for {Model_VAR_NAME}, got {xy_dims}')
    y_dim, x_dim = xy_dims

    Model_sub = Model_ds.isel({
        z_dim: MODEL_SURFACE_LAYER_INDEX,
        y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
        x_dim: slice(X_SLICE[0], X_SLICE[1])
    })

# ---------------------------------------------------------
# 2. Compute spatial mean
# ---------------------------------------------------------
model_series = Model_sub.mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Day of year
model_series['doy'] = model_series[time_dim].dt.dayofyear

# ---------------------------------------------------------
# 3. Observations
# ---------------------------------------------------------
if Obs_VAR_NAME not in measurement_df.columns:
    raise KeyError(f"Column {Obs_VAR_NAME} not found in the file.")

obs_df = measurement_df[['timestamp', Obs_VAR_NAME]].dropna(subset=['timestamp', Obs_VAR_NAME]).copy()
obs_df['timestamp'] = pd.to_datetime(obs_df['timestamp'], dayfirst=True, errors='coerce')
obs_df = obs_df.sort_values('timestamp')

if obs_df.empty:
    raise ValueError('No measurements found.')

obs_df['doy'] = obs_df['timestamp'].dt.dayofyear

# ---------------------------------------------------------
# 4. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.scatter(
    obs_df['doy'],
    #obs_df['POC'] * 1000, # for POC to be mg/m3
    obs_df[Obs_VAR_NAME],
    s=22,
    color='tab:orange',
    alpha=0.85,
    label=f'Observed {Obs_VAR_NAME}',
    zorder=3,
)

ax.plot(
    model_series['doy'],
    model_series.values,
    lw=2.0,
    color='tab:blue',
    label=f'Model {Model_VAR_NAME}',
    zorder=4,  # above the scatter (zorder=3)
)


Var_units = Model_ds.attrs.get('units', '')
ax.set_ylabel(f'{Obs_VAR_NAME} [{Var_units}]')
ax.set_xlabel('Day of Year')
ax.set_title(
    f'Model vs observed {Obs_VAR_NAME} | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Number of observation points: {len(obs_df)}')


In [ ]:
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.


Obs_VAR_NAME = 'TSM'
Model_VAR_NAME = 'ESS'

#Obs_VAR_NAME = 'Daily_PP'
#Model_VAR_NAME = 'netPPm2'

#Obs_VAR_NAME = 'POC'
#Model_VAR_NAME = 'R6c'

#Obs_VAR_NAME = 'NH4'
#Model_VAR_NAME = 'N4n'

#Obs_VAR_NAME = 'Chl'
#Model_VAR_NAME = 'Chla'

if Model_VAR_NAME not in ds.variables:
    raise KeyError(f"Variable '{Model_VAR_NAME}' not found. Available: {sorted(ds.data_vars)}")

Model_ds = ds[Model_VAR_NAME].squeeze(drop=True)
time_dim = _find_time_dim(Model_ds)

z_dim = _find_vertical_dim(Model_ds, time_dim)   # <-- may be None for 2D variables

# ---------------------------------------------------------
# 1. Handle 2D vs 3D variables
# ---------------------------------------------------------
if z_dim is None:
    # 2D variable: dims = (time, y, x)
    spatial_dims = [d for d in Model_ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected 2D variable with dims (time,y,x), got {Model_ds.dims}")
    y_dim, x_dim = spatial_dims
    Model_sub = Model_ds.isel({y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
                               x_dim: slice(X_SLICE[0], X_SLICE[1])})
else:
    # 3D variable: dims = (time, z, y, x)
    xy_dims = [d for d in Model_ds.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims for {Model_VAR_NAME}, got {xy_dims}')
    y_dim, x_dim = xy_dims

    Model_sub = Model_ds.isel({
        z_dim: MODEL_SURFACE_LAYER_INDEX,
        y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
        x_dim: slice(X_SLICE[0], X_SLICE[1])
    })

# ---------------------------------------------------------
# 2. Compute spatial mean
# ---------------------------------------------------------
model_series = Model_sub.mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Day of year
model_series['doy'] = model_series[time_dim].dt.dayofyear

# ---------------------------------------------------------
# 3. Observations
# ---------------------------------------------------------
if Obs_VAR_NAME not in measurement_df.columns:
    raise KeyError(f"Column {Obs_VAR_NAME} not found in the file.")

obs_df = measurement_df[['timestamp', Obs_VAR_NAME]].dropna(subset=['timestamp', Obs_VAR_NAME]).copy()
obs_df['timestamp'] = pd.to_datetime(obs_df['timestamp'], dayfirst=True, errors='coerce')
obs_df = obs_df.sort_values('timestamp')

if obs_df.empty:
    raise ValueError('No measurements found.')

obs_df['doy'] = obs_df['timestamp'].dt.dayofyear

# ---------------------------------------------------------
# 4. Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 4.8))

ax.scatter(
    obs_df['doy'],
    obs_df[Obs_VAR_NAME]*1e3, # for POC to be mg/m3
    s=22,
    color='tab:orange',
    alpha=0.85,
    label=f'Observed {Obs_VAR_NAME}',
    zorder=3,
)

ax.plot(
    model_series['doy'],
    model_series.values,
    lw=2.0,
    color='tab:blue',
    label=f'Model {Model_VAR_NAME}',
    zorder=4,  # above the scatter (zorder=3)
)


Var_units = Model_ds.attrs.get('units', '')
ax.set_ylabel(f'{Obs_VAR_NAME} [{Var_units}]')
ax.set_xlabel('Day of Year')
ax.set_title(
    f'Model vs observed {Obs_VAR_NAME} | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Number of observation points: {len(obs_df)}')
